[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/13_svm_and_kernels/03_kernel_trick.ipynb)

# Bài 03 — Kernel trick: SVM phi tuyến

**🎯 Mục tiêu**
- Feature mapping $\phi(\mathbf x)$: dữ liệu không tách được tuyến tính trong không gian gốc có thể tách được trong không gian **nhiều chiều hơn**.
- Bài toán **đối ngẫu** (dual) của SVM chỉ cần **tích vô hướng** $\mathbf x_i^\top\mathbf x_j$ → thay bằng **kernel** $K(\mathbf x_i, \mathbf x_j) = \phi(\mathbf x_i)^\top\phi(\mathbf x_j)$ mà không cần tính $\phi$.
- Các kernel: linear, polynomial, **RBF (Gaussian)**; ý nghĩa của $\gamma$ và $C$.
- Kernel SVM bằng SMO (`KernelSVMScratch`) và scikit-learn.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/13_svm_and_kernels

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from ml_course.datasets import make_classification_2d
from ml_course.visualization import plot_decision_boundary, set_style

set_style()

## 1. Feature mapping: nâng chiều để tách được

Dữ liệu "vòng tròn": không đường thẳng nào tách được. Thêm feature $x_1^2 + x_2^2$ (bình phương khoảng cách tới gốc) → trong không gian 3D, một **mặt phẳng** tách hoàn hảo:

In [ ]:
X, y = make_classification_2d("circles", n=300, noise=0.15, seed=0)
r2 = (X**2).sum(1)
fig = plt.figure(figsize=(14, 5))
ax1 = fig.add_subplot(1, 2, 1)
ax1.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", s=15)
ax1.set_aspect("equal")
ax1.set_title("Không gian gốc (2D): không tách tuyến tính được")
ax2 = fig.add_subplot(1, 2, 2, projection="3d")
ax2.scatter(X[:, 0], X[:, 1], r2, c=y, cmap="coolwarm", s=15)
ax2.set_zlabel("x1² + x2²")
ax2.set_title("Thêm chiều x1² + x2²: tách bằng mặt phẳng z ≈ 0.5")
plt.show()

Vấn đề: polynomial bậc $p$ của $d$ feature có $\binom{d + p}{p}$ chiều — bùng nổ nhanh (d = 100, p = 3: ~177 000 chiều). Kernel trick giải quyết chuyện này.

## 2. Bài toán đối ngẫu chỉ cần tích vô hướng

Dùng nhân tử Lagrange, bài toán soft-margin SVM tương đương (dual):

$$\max_{\alpha}\ \sum_i \alpha_i - \frac12\sum_{i,j}\alpha_i\alpha_j y_i y_j\,\mathbf x_i^\top\mathbf x_j \quad\text{s.t.}\quad 0 \le \alpha_i \le C,\ \ \sum_i\alpha_i y_i = 0$$

và hàm dự đoán:

$$f(\mathbf x) = \sum_i \alpha_i y_i\,\mathbf x_i^\top\mathbf x + b$$

- $\alpha_i > 0$ **chỉ** với support vectors → tổng thưa.
- Dữ liệu chỉ xuất hiện qua **tích vô hướng** $\mathbf x_i^\top\mathbf x_j$. Thay bằng $K(\mathbf x_i, \mathbf x_j) = \phi(\mathbf x_i)^\top\phi(\mathbf x_j)$ → SVM trong không gian $\phi$ **mà không bao giờ tính $\phi$**. Đó là **kernel trick**.

## 3. Kiểm chứng: polynomial kernel bậc 2

Với $\mathbf x \in \mathbb R^2$, $K(\mathbf a, \mathbf b) = (\mathbf a^\top\mathbf b + 1)^2$ bằng đúng tích vô hướng của

$$\phi(\mathbf x) = \left(1,\ \sqrt2 x_1,\ \sqrt2 x_2,\ x_1^2,\ x_2^2,\ \sqrt2 x_1x_2\right)$$

In [ ]:
def phi(x):
    x1, x2 = x
    return np.array([1, np.sqrt(2) * x1, np.sqrt(2) * x2, x1**2, x2**2, np.sqrt(2) * x1 * x2])


a, b = np.array([1.0, 2.0]), np.array([3.0, -1.0])
print(f"K(a, b) = (aᵀb + 1)² = {(a @ b + 1) ** 2:.4f}")
print(f"φ(a)ᵀφ(b)           = {phi(a) @ phi(b):.4f}")

Tính kernel: $O(d)$ phép nhân. Tính $\phi$ rồi nhân: $O(d^p)$. Với **RBF kernel**, $\phi$ có **vô hạn** chiều — không thể tính trực tiếp, nhưng kernel thì rất rẻ.

## 4. Các kernel phổ biến

| Kernel | $K(\mathbf a, \mathbf b)$ | Ghi chú |
|---|---|---|
| Linear | $\mathbf a^\top\mathbf b$ | = SVM tuyến tính |
| Polynomial | $(\gamma\,\mathbf a^\top\mathbf b + r)^p$ | tương tác bậc ≤ p |
| **RBF (Gaussian)** | $\exp(-\gamma\lVert\mathbf a - \mathbf b\rVert^2)$ | mặc định; "độ giống" giảm theo khoảng cách; $\phi$ vô hạn chiều |
| Sigmoid | $\tanh(\gamma\,\mathbf a^\top\mathbf b + r)$ | ít dùng, không luôn hợp lệ |

Với RBF, $f(\mathbf x) = \sum_i \alpha_i y_i e^{-\gamma\lVert\mathbf x_i - \mathbf x\rVert^2} + b$: tổng các "quả đồi Gaussian" đặt tại support vectors — rất giống **KNN có trọng số** (chương 08) nhưng chỉ dùng support vectors và trọng số được **học**.

In [ ]:
Xm, ym = make_classification_2d("moons", n=300, noise=0.25, seed=1)
Xm = StandardScaler().fit_transform(Xm)
fig, axes = plt.subplots(1, 3, figsize=(19, 5))
for ax, (name, model) in zip(axes, [("linear", SVC(kernel="linear", C=1)),
                                    ("polynomial bậc 3", SVC(kernel="poly", degree=3, C=1, coef0=1)),
                                    ("RBF", SVC(kernel="rbf", C=1, gamma=1))]):
    model.fit(Xm, ym)
    plot_decision_boundary(model, Xm, ym, ax=ax)
    ax.scatter(*Xm[model.support_].T, s=60, facecolors="none", edgecolors="k", lw=0.6)
    ax.set_title(f"kernel {name}: train acc = {model.score(Xm, ym):.2f}, {len(model.support_)} SV")
plt.show()

## 5. RBF: vai trò của γ và C

- $\gamma$ lớn → mỗi "quả đồi" **hẹp** → ranh giới uốn éo quanh từng điểm (overfit). $\gamma$ nhỏ → đồi rộng → ranh giới mượt (gần tuyến tính).
- $C$ như bài 02: lớn → ít chấp nhận vi phạm.

In [ ]:
X_val, y_val = make_classification_2d("moons", n=2000, noise=0.25, seed=2)
scaler = StandardScaler().fit(make_classification_2d("moons", n=300, noise=0.25, seed=1)[0])
X_raw, y_raw = make_classification_2d("moons", n=300, noise=0.25, seed=1)
Xs, Xv = scaler.transform(X_raw), scaler.transform(X_val)
fig, axes = plt.subplots(3, 3, figsize=(16, 14))
for i, gamma in enumerate([0.1, 1, 30]):
    for j, C in enumerate([0.1, 1, 100]):
        model = SVC(kernel="rbf", gamma=gamma, C=C).fit(Xs, y_raw)
        plot_decision_boundary(model, Xs, y_raw, ax=axes[i, j])
        axes[i, j].set_title(f"γ = {gamma}, C = {C}: train {model.score(Xs, y_raw):.2f} | val {model.score(Xv, y_val):.2f}", fontsize=10)
plt.tight_layout()
plt.show()

## 6. Kernel SVM from scratch (SMO)

Bài toán dual được giải hiệu quả bằng **SMO** (Sequential Minimal Optimization, Platt 1998): mỗi bước chọn **hai** $\alpha_i, \alpha_j$ và tối ưu chúng giải tích (ràng buộc $\sum\alpha_i y_i = 0$ buộc phải đổi theo cặp). Phiên bản rút gọn có trong `ml_course.algorithms.KernelSVMScratch`:

In [ ]:
from ml_course.algorithms import KernelSVMScratch

ours = KernelSVMScratch(C=1.0, kernel="rbf", gamma=1.0, seed=0).fit(Xs, y_raw)
sk = SVC(kernel="rbf", C=1.0, gamma=1.0).fit(Xs, y_raw)
agree = np.mean(ours.predict(Xv) == sk.predict(Xv))
print(f"SMO scratch: {len(ours.support_)} SV, val acc = {np.mean(ours.predict(Xv) == y_val):.3f}")
print(f"sklearn SVC: {len(sk.support_)} SV, val acc = {sk.score(Xv, y_val):.3f}")
print(f"dự đoán trùng nhau trên {agree:.1%} điểm validation")

## 7. Kernel là "độ giống nhau"

Một kernel hợp lệ (Mercer) tương ứng với ma trận Gram $K_{ij} = K(\mathbf x_i, \mathbf x_j)$ **nửa xác định dương**. Có thể thiết kế kernel cho dữ liệu đặc biệt: chuỗi (string kernel), đồ thị, tập hợp... — ý tưởng "học bằng độ giống nhau" này xuất hiện lại trong Gaussian Process (chương 23) và attention (chương 31).

In [ ]:
K = np.exp(-1.0 * ((Xs[:60, None, :] - Xs[None, :60, :]) ** 2).sum(-1))
order = np.argsort(y_raw[:60])
plt.imshow(K[np.ix_(order, order)], cmap="viridis")
plt.colorbar(label="K(xi, xj)")
plt.title("Ma trận RBF kernel (60 điểm, sắp xếp theo lớp)")
plt.show()
print("trị riêng nhỏ nhất của K:", np.linalg.eigvalsh(K).min().round(8), "(≥ 0: nửa xác định dương)")

## 🧠 Kiểm tra nhanh

1. Vì sao dạng dual của SVM cho phép dùng kernel trick?
2. Tính $K(\mathbf a, \mathbf b)$ với RBF, $\gamma = 0.5$, $\lVert\mathbf a - \mathbf b\rVert^2 = 2$.
3. RBF SVM overfit — nên tăng hay giảm $\gamma$? $C$?
4. RBF SVM giống và khác KNN ở điểm nào?

## 📌 Tóm tắt

- Dual chỉ cần tích vô hướng → thay bằng kernel → SVM trong không gian đặc trưng nhiều (thậm chí vô hạn) chiều.
- RBF: $\gamma$ điều khiển độ "hẹp" của ảnh hưởng mỗi support vector; $C$ điều khiển mức chấp nhận vi phạm.
- Dự đoán = tổng có trọng số của kernel giữa điểm mới và support vectors.

## 📚 Tài liệu tham khảo

- Schölkopf & Smola (2002) — *Learning with Kernels*
- Platt (1998) — *Sequential Minimal Optimization*
- Vũ Hữu Tiệp — [Kernel SVM](https://machinelearningcoban.com/2017/04/22/kernelsmv/)